# 00 — Experiment Readiness

This notebook validates manifest metadata and participant-level split isolation. Its examples are synthetic infrastructure checks, not evaluation evidence or actual human data. No inference, MLflow access, or cloud resource is created.


In [ ]:
REQUIRED = {'clip_id', 'sha256', 'participant_id', 'split', 'exercise', 'consent_scope', 'annotation_version'}
EXERCISES = {'squat', 'biceps_curl', 'lateral_raise'}

def validate_manifest(rows):
    errors, seen, participant_splits = [], set(), {}
    for i, row in enumerate(rows):
        missing = REQUIRED - row.keys()
        if missing:
            errors.append(f'row {i}: missing {sorted(missing)}')
            continue
        if row['clip_id'] in seen:
            errors.append(f'row {i}: duplicate clip')
        seen.add(row['clip_id'])
        if row['exercise'] not in EXERCISES:
            errors.append(f'row {i}: unsupported exercise')
        if row['split'] not in {'development', 'heldout'}:
            errors.append(f'row {i}: invalid split')
        if not row['consent_scope']:
            errors.append(f'row {i}: missing usage scope')
        digest = row['sha256']
        if not isinstance(digest, str) or len(digest) != 64 or any(c not in '0123456789abcdef' for c in digest):
            errors.append(f'row {i}: invalid checksum format')
        participant_splits.setdefault(row['participant_id'], set()).add(row['split'])
    for participant, splits in participant_splits.items():
        if len(splits) > 1:
            errors.append(f'participant {participant}: split leakage')
    return errors


In [ ]:
synthetic = [{'clip_id': 'fixture-a', 'sha256': 'a' * 64, 'participant_id': 'synthetic-1', 'split': 'development', 'exercise': 'squat', 'consent_scope': 'synthetic-no-person', 'annotation_version': 'fixture-v1'}]
assert validate_manifest(synthetic) == []
leaking = synthetic + [{**synthetic[0], 'clip_id': 'fixture-b', 'split': 'heldout'}]
assert any('split leakage' in e for e in validate_manifest(leaking))
assert any('missing' in e for e in validate_manifest([{}]))
print('Synthetic manifest checks passed; real dataset evaluation remains pending.')


## Before a real experiment

Verify actual file hashes and media permissions, inspect annotation quality, freeze participant splits, record the code/configuration revision, and connect the approved managed MLflow tracking server. Passing metadata checks does not prove consent, annotation correctness, adequate coverage, or model quality. Move this validation into the evaluation source package when it is scaffolded; later notebooks should import that shared implementation.
